# Run feedback — rating and commenting on finished runs

The reviewing layer: how a human marks which conversations went well and which need attention,
at the level of a whole turn or of a single event inside it.

Feedback lives on `client.runs`, not a separate resource. Two facts shape everything below:

- **The run must be finished.** Feedback on an in-flight run is refused with **409**.
- **Every write returns the whole list**, not just your record — so a UI can render aggregate
  counts and author names without a refetch.

**See also:** [Run feedback guide](../docs/guides/run_feedback.md)

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

## 1. A finished run to review

Two static nodes, so it reaches a terminal state on its own.

In [ ]:
import _bootstrap  # noqa: F401

import interactly_configs as ic
from interactly import AsyncWorkflowClient, ConflictError, WorkflowCommand

client = AsyncWorkflowClient()

hello = ic.SayStaticMessageNodeConfig(
    name="Hello", is_start=True,
    static_messages_config=ic.StaticMessagesConfig(static_messages=["Hello — how can I help?"]),
)
bye = ic.SayStaticMessageNodeConfig(
    name="Bye",
    static_messages_config=ic.StaticMessagesConfig(static_messages=["Goodbye."]),
)

workflow = await client.workflows.create_from_config(
    ic.WorkflowConfigFullyHydrated(
        workflow_config=ic.WorkflowConfig(name="NB21: Run feedback"),
        node_configs=[hello, bye],
        edge_configs=[ic.DirectEdgeConfig(source_node_logical_id=hello.logical_id,
                                          destination_node_logical_id=bye.logical_id)]),
    name="NB21: Run feedback")
WORKFLOW_ID = workflow.id

RUN_ID = None
async with client.runs.stream(workflow_id=WORKFLOW_ID, command=WorkflowCommand.START) as stream:
    async for event in stream:
        if getattr(event, "workflow_run_id", None):
            RUN_ID = event.workflow_run_id
        if event.type == "assistant_response":
            print("🤖", event.output)
        if event.is_terminal():
            break

run = await client.runs.get(RUN_ID)
print(f"\nrun {RUN_ID}  status={run.status}  turns={len(run.input_output_pairs)}")

## 2. Ratings

Three values, deliberately. A plain up/down cannot distinguish "fine" from "exactly right", and
that distinction is what makes a review set useful later.

`ic.RATING_SCORES` maps them to numeric weights for aggregation — but the **string** is what gets
stored, never the score. A score is derivable, and keeping the name means new values can be added
without re-interpreting old records.

In [ ]:
print("values:", [v.value for v in ic.RatingValue])
print("scores:", {k.value: v for k, v in ic.RATING_SCORES.items()})

result = await client.runs.set_turn_rating(RUN_ID, turn_index=0, value=ic.RatingValue.UP)

# Every write returns the FULL list for the mutated target — everyone's, not just yours.
for rating in result.ratings:
    author = result.feedback_users.get(rating["createdBy"])
    who = f"{author.firstName} {author.lastName}" if author else "unknown"
    print(f"  {rating['value']:10s} by {who}")

### At most one rating per user per target

Re-rating **replaces** your record rather than appending — `createdBy` is effectively the identity
key within a target's rating list, so there is no "remove then add" dance.

In [ ]:
again = await client.runs.set_turn_rating(RUN_ID, 0, ic.RatingValue.STRONG_UP)
print(f"after re-rating: {len(again.ratings)} rating(s) — "
      f"{[r['value'] for r in again.ratings]}")

# Both the enum and the plain string are accepted.
await client.runs.set_turn_rating(RUN_ID, 0, "up")
print("plain string accepted too")

## 3. Comments

Content is **trimmed** before storage, and a whitespace-only comment is refused rather than being
stored as a comment nobody can see.

The cap is `ic.MAX_COMMENT_LENGTH` — roughly ten dense paragraphs. Comments are embedded in the run
document, which already carries every event of every turn and is the thing under storage pressure;
this is not a document store.

In [ ]:
print("MAX_COMMENT_LENGTH:", ic.MAX_COMMENT_LENGTH)

result = await client.runs.add_turn_comment(RUN_ID, turn_index=0, content="  Missed the callback number  ")
for comment in result.comments:
    print(f"  {comment['logical_id'][:16]}… {comment['content']!r}   ← trimmed")

COMMENT_ID = result.comments[0]["logical_id"]

### Not every write returns the whole list

`set_turn_rating`, `set_event_rating` and `add_turn_comment` return a `RunFeedbackResponse` —
the full list plus `feedback_users`. The two **older** endpoints, `add_comment` (run-level) and
`add_event_comment`, predate that and return a single `RunComment`. Check the return type rather
than assuming.

In [ ]:
# Run-level, for a note about the whole conversation. Returns a single RunComment.
run_comment = await client.runs.add_comment(RUN_ID, content="Reviewed — no action needed")
print(f"run-level comment: {type(run_comment).__name__}  {run_comment.content!r}")

# ...and blank content is refused.
try:
    await client.runs.add_turn_comment(RUN_ID, 0, content="   ")
    print("❗ blank comment unexpectedly accepted")
except Exception as exc:
    print(f"✅ blank comment refused: {type(exc).__name__}: {str(exc)[:120]}")

## 4. `feedback_users` — names without a directory lookup

`FeedbackUser` carries only `id`, `firstName`, `lastName`, `email`. A run response should not
become an access path to the user directory.

Ids that no longer resolve are simply **absent** from the map, so look them up with `.get()`
rather than `[]` — which is what the rating loop in section 2 does.

> **Verified on dev, 2026-08-07:** the map is populated on the **write** responses
> (`RunFeedbackResponse.feedback_users`), but `client.runs.get(...)` returns it **empty** even
> when the run carries ratings and comments. `Run.feedback_users` exists on the model and
> defaults to `{}`, so this fails silently rather than raising. Resolve author names from the
> write response, or from your own directory.

In [ ]:
from interactly.types.runs.run import FeedbackUser

print("FeedbackUser fields:", list(FeedbackUser.model_fields))

run = await client.runs.get(RUN_ID)
print(f"\nfeedback_users on a fetched run: {run.feedback_users or '{} (empty — see above)'}")

# The write response is where the names actually are.
res = await client.runs.set_turn_rating(RUN_ID, 0, ic.RatingValue.UP)
print(f"feedback_users on the write response: {len(res.feedback_users)} entry(ies)")
for rating in res.ratings:
    author = res.feedback_users.get(rating["createdBy"])
    print(f"  {rating['value']:10s} by {author.firstName + ' ' + author.lastName if author else 'unknown'}")

## 5. Reading feedback back off the run

Turn-level feedback lives on each input/output pair; event-level feedback lives on the events
inside `pair.run_output.events`.

In [ ]:
run = await client.runs.get(RUN_ID)

for i, pair in enumerate(run.input_output_pairs):
    if pair.ratings or pair.comments:
        print(f"turn {i}: ratings={[r.value for r in pair.ratings]}")
        for comment in pair.comments:
            print(f"   — {comment.content!r}")

## 6. Event-level feedback

For finer-grained review: one bad assistant response inside an otherwise fine turn.

In [ ]:
# Find an event to rate.
pair = run.input_output_pairs[0]
event_logical_id = None
for ev in (pair.run_output.events if pair.run_output else []):
    ev_id = ev.get("logical_id") if isinstance(ev, dict) else getattr(ev, "logical_id", None)
    ev_type = ev.get("type") if isinstance(ev, dict) else getattr(ev, "type", None)
    if ev_type == "assistant_response" and ev_id:
        event_logical_id = ev_id
        break

if event_logical_id:
    res = await client.runs.set_event_rating(RUN_ID, event_logical_id, ic.RatingValue.DOWN)
    print(f"rated event {event_logical_id[:24]}… → {[r['value'] for r in res.ratings]}")

    # add_event_comment is one of the two older endpoints — a single RunComment back.
    ev_comment = await client.runs.add_event_comment(RUN_ID, event_logical_id, content="This is the bad reply")
    print(f"commented: {type(ev_comment).__name__}  {ev_comment.content!r}")
else:
    print("no assistant_response event found to rate")

## 7. Removing feedback

In [ ]:
await client.runs.delete_turn_comment(RUN_ID, turn_index=0, comment_logical_id=COMMENT_ID)
after = await client.runs.delete_turn_rating(RUN_ID, turn_index=0)
print(f"turn ratings now: {[r['value'] for r in after.ratings] or 'none'}")

if event_logical_id:
    await client.runs.delete_event_rating(RUN_ID, event_logical_id)
    print("event rating removed")

## 8. The 409: an in-flight run cannot be reviewed

Check `run.status` first, or drive the run to a terminal state before reviewing it. The run below
is deliberately left waiting for input.

In [ ]:
llm = ic.LLMGroupConfig(llms=[ic.OpenAILLMConfig(model=ic.OPENAIModel.GPT_4_1_MINI, max_tokens=60)])
chatty = ic.SayLLMNodeConfig(
    name="Chat", is_start=True, wait_for_user_message=True, self_loop=True,
    main_response_config=ic.PromptConfig(prompt="Say hello in one short sentence."), llms_config=llm)

wf2 = await client.workflows.create_from_config(
    ic.WorkflowConfigFullyHydrated(
        workflow_config=ic.WorkflowConfig(name="NB21: in-flight"),
        node_configs=[chatty], edge_configs=[]),
    name="NB21: in-flight")

inflight = await client.runs.execute(wf2.id, command=WorkflowCommand.START)
print(f"run status: {inflight.status}")

try:
    await client.runs.set_turn_rating(inflight.run_id, 0, ic.RatingValue.UP)
    print("❗ unexpectedly accepted")
except ConflictError as exc:
    print(f"✅ refused with 409:\n   {str(exc)[:200]}")

await client.workflows.delete(wf2.id)

## 9. A review loop

Rate what you find, then re-run the ones that went wrong — the two halves of this notebook and
[notebook 18](18_reruns_and_replay.ipynb) meeting.

In [ ]:
page = await client.runs.list(workflow_id=WORKFLOW_ID, size=20)

async for r in page:
    detail = await client.runs.get(r.id)
    if detail.error:
        await client.runs.set_turn_rating(r.id, 0, ic.RatingValue.DOWN)
        await client.runs.add_turn_comment(r.id, 0, f"Failed: {detail.error}")

        # Only WS-driven runs can be re-run — ask, do not assume.
        turns = await client.reruns.rerunnable_turns(r.id)
        if any(t.rerunnable for t in turns.turns):
            await client.reruns.execute(r.id, turn_index=0)
            print(f"  re-ran {r.id}")
    else:
        print(f"  {r.id}  {r.status}  (nothing to flag)")

## Cleanup

In [ ]:
await client.workflows.delete(WORKFLOW_ID)
await client.close()
print("Deleted", WORKFLOW_ID)

## See also

- [`18_reruns_and_replay.ipynb`](18_reruns_and_replay.ipynb) — replaying the runs that went wrong
- [`14_pagination_and_filtering.ipynb`](14_pagination_and_filtering.ipynb) — finding runs worth reviewing
- [Run feedback guide](../docs/guides/run_feedback.md)

### Gotchas

- **Feedback needs a finished run** — an in-flight run gives you 409.
- **Re-rating replaces**; it does not append. There is no duplicate to clean up.
- **Blank comments are refused**, and content is trimmed before storage.
- **`feedback_users` can miss an id** if the user no longer resolves. Use `.get()`.
- **`turn_index` is the 0-based position** in `input_output_pairs`, not a turn id.
- **`result.ratings` is everyone's**, not just yours — filter by `createdBy` for your own.
- **`add_comment` / `add_event_comment` return a single `RunComment`**, not the full list — only
  the rating endpoints and `add_turn_comment` return a `RunFeedbackResponse`.
- **`feedback_users` is empty on a fetched run** today; read it off the write response.